# train_controls

Trains the homogeneous controls for the 10 original seeds: the fixed-tau ladder (2, 3.5, 15, 35 ms, plus a 200 ms rung for seed 202) and the scaled-init nets. Checkpoints land in checkpoints/controls.

## config

In [ ]:
from pathlib import Path
import sys, time
import numpy as np
import torch

BASE = Path("..")
# from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/Clean Repo")

sys.path.insert(0, str((BASE / "code").resolve()))
import train_common as tc

DEVICE = tc.DEVICE
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

TASK_KEY = "4class"
MEM_FAMILY = "lognormal"
SEEDS = [101, 202, 210, 340, 440, 550, 660, 710, 820, 930]

tc.BASE_PRMS["nb_recurrent"] = 64
tc.BASE_PRMS["batch_size"] = 256
tc.BASE_PRMS["nb_epochs"] = 50
tc.BASE_PRMS["early_stop_patience"] = 10
tc.BASE_PRMS["optimizer"] = "adam"
tc.BASE_PRMS["weight_decay"] = 1e-4
tc.BASE_PRMS["lr"] = 2e-3
tc.BASE_PRMS["lr_ab"] = 2e-3
tc.BASE_PRMS["lr_patience"] = 5
tc.BASE_PRMS["lr_factor"] = 0.5
tc.BASE_PRMS["lr_min"] = 5e-5
tc.BASE_PRMS["compile_model"] = False

# (tau_ms, seeds, run_label)
FIXED_JOBS = [
    (15.0, SEEDS, "fixed_tau_sweep"),
    (35.0, SEEDS, "fixed_tau_sweep"),
    (2.0, SEEDS, "short_fixed_tau"),
    (3.5, SEEDS, "short_fixed_tau"),
    (200.0, [202], "long_fixed_tau"),
]

# per-seed init weight scale (input+recurrent) matching hom init rate to het trained rate;
SCALE_C = {101: 6.522, 202: 6.87, 210: 6.419, 340: 5.057, 440: 8.0,
           550: 7.595, 660: 3.776, 710: 4.502, 820: 8.0, 930: 6.897}
WHICH_WEIGHTS = {"input_w": True, "recurrent_v": True, "output_w": False}
W_HID, V_HID, W_OUT = "network.0.w", "network.0.v", "network.1.w"

def tau_tag(t):
    return f"{t:g}".replace(".", "p")

OUT_FIXED = "4class_seed{s}_hom_fixed{tag}ms_lognormal.pt"
OUT_SCALED = "4class_seed{s}_hom_scaledinit_inputrec_lognormal.pt"

CKPT_OUT = BASE / "checkpoints" / "controls"
CKPT_OUT.mkdir(parents=True, exist_ok=True)
print("device:", DEVICE, "| output:", CKPT_OUT.resolve())

## helpers

In [ ]:
def build_fixed_hom(seed, tau_mem_ms):
    # non-tau weights synced from the seed's het twin, hidden tau_mem frozen at the given value
    het_init, _ = tc.build_heterogeneous_model(TASK_KEY, seed)
    prms = tc._base_prms_for_task(TASK_KEY, seed)
    prms.update({"het_ab": 0, "train_ab": 0, "train_hom_ab": 0})
    prms.update({"dtype": torch.float, "device": DEVICE, "cuda": DEVICE.type == "cuda"})
    tc.set_seed(seed)
    model = tc.RSNN(prms, rec=True).to(DEVICE)
    tc.sync_non_tau_parameters(het_init, model)
    tc.set_layer_taus_ms(model.network[0], tau_syn_ms=tc.DEFAULT_HIDDEN_TAU_SYN_MS,
                         tau_mem_ms=float(tau_mem_ms), time_step=prms["time_step"])
    tc.set_layer_taus_ms(model.network[1], tau_syn_ms=tc.DEFAULT_OUTPUT_TAU_SYN_MS,
                         tau_mem_ms=tc.DEFAULT_OUTPUT_TAU_MEM_MS, time_step=prms["time_step"])
    assert sum(p.requires_grad for n, p in model.named_parameters()
               if ("alpha" in n or "beta" in n)) == 0
    got = tc.decay_to_tau_ms(model.network[0].beta.detach().cpu().numpy().ravel(), prms["time_step"])
    assert np.allclose(got, tau_mem_ms, atol=1e-6) and np.unique(np.round(got, 6)).size == 1
    return model, prms

def scaled_state(base_state, c, which=WHICH_WEIGHTS):
    s = {k: (v.clone() if torch.is_tensor(v) else v) for k, v in base_state.items()}
    if which.get("input_w"):
        s[W_HID] = s[W_HID] * float(c)
    if which.get("recurrent_v"):
        s[V_HID] = s[V_HID] * float(c)
    if which.get("output_w"):
        s[W_OUT] = s[W_OUT] * float(c)
    return s

## data

In [ ]:
train_cache, test_cache = tc.load_default_caches()

## fixed-tau ladder

In [ ]:
for tau, seeds, run_label in FIXED_JOBS:
    tag = tau_tag(tau)
    for seed in seeds:
        out_path = CKPT_OUT / OUT_FIXED.format(s=seed, tag=tag)
        if out_path.exists():
            print(f"tau={tau:g}ms seed {seed}: exists, skip")
            continue
        print(f"\ntau={tau:g}ms seed {seed}: training")
        model, prms = build_fixed_hom(seed, tau)
        t0 = time.time()
        history = tc.train_experiment(model, prms, train_cache, test_cache)
        elapsed = time.time() - t0
        tc.save_model_checkpoint(
            out_path, model, history, elapsed, prms,
            extra={"label": out_path.stem, "task": TASK_KEY, "task_key": TASK_KEY,
                   "pair_seed": seed, "pair_role": "homogeneous_fixed_tau",
                   "run_label": run_label, "mem_distribution_family": MEM_FAMILY,
                   "control_type": "hom_tau_frozen_fixed", "fixed_tau_mem_ms": float(tau)})
        print(f"saved {out_path.name} acc={history['test_acc'][-1]:.3f} ({elapsed/60:.1f} min)")

## scaled-init controls

In [ ]:
for seed in SEEDS:
    out_path = CKPT_OUT / OUT_SCALED.format(s=seed)
    if out_path.exists():
        print(f"scaled seed {seed}: exists, skip")
        continue
    c = SCALE_C[seed]
    pair = tc.build_seeded_pair(master_seed=seed, task_key=TASK_KEY, mem_distribution_family=MEM_FAMILY)
    assert pair["metadata"]["linear_sync_verified"]
    hom = pair["hom_model"]
    hom_prms = dict(pair["hom_prms"])
    hom_prms["weight_decay"] = 1e-4
    hom_prms.update({"dtype": torch.float, "device": DEVICE, "cuda": DEVICE.type == "cuda"})
    assert hom_prms.get("train_hom_ab") == 1 and hom_prms.get("het_ab") == 0
    hom.load_state_dict(scaled_state(hom.state_dict(), c))
    tau0 = tc.decay_to_tau_ms(hom.network[0].beta.detach().cpu().numpy().ravel(), hom_prms["time_step"])
    assert np.unique(np.round(tau0, 6)).size == 1
    print(f"\nscaled seed {seed}: c={c:.3f} init tau={tau0[0]:.1f} ms")
    t0 = time.time()
    history = tc.train_experiment(hom, hom_prms, train_cache, test_cache)
    elapsed = time.time() - t0
    tc.save_model_checkpoint(
        out_path, hom, history, elapsed, hom_prms,
        extra={"label": out_path.stem, "task": TASK_KEY, "task_key": TASK_KEY,
               "pair_seed": seed, "pair_role": "homogeneous_scaled_init",
               "run_label": "scaled_init", "mem_distribution_family": MEM_FAMILY,
               "control_type": "hom_scaled_init", "init_weight_scale": float(c),
               "scale_which_weights": WHICH_WEIGHTS,
               "scale_target_metric": "het_trained_mean_rate"})
    print(f"saved {out_path.name} acc={history['test_acc'][-1]:.3f} ({elapsed/60:.1f} min)")